### Notebook to Search and Filter Networking Nature Data for Lime Keywords

In [25]:
import pandas as pd
import re
from pathlib import Path

In [26]:
import urllib.parse

# --- Keywords ---
WATER_KEYWORDS = [
    "shore", "ocean", "sea", "river", "lake", "pond", "stream", "creek",
    "wetland", "reservoir", "estuary", "gulf", "bay",
    "body of water", "bodies of water"
]

# Per-keyword patterns (word boundaries; multi-word phrases work naturally)
kw_patterns = {
    kw: re.compile(r"\b" + re.escape(kw) + r"\b", flags=re.IGNORECASE)
    for kw in WATER_KEYWORDS
}

# Combined filter pattern (non-capturing group avoids UserWarning)
combined_pattern = re.compile(
    r"\b(?:" + "|".join(re.escape(kw) for kw in WATER_KEYWORDS) + r")\b",
    flags=re.IGNORECASE
)

# Extracts institution code and volume ID from source_file name
# e.g. "blackwood_v24_1828_inu-30000080770807-1746739609.txt" -> ("inu", "30000080770807")
source_file_pattern = re.compile(r'_([a-z]+)-(\d+)-\d+\.txt$')

def make_hathitrust_link(source_file, kw_counts):
    m = source_file_pattern.search(str(source_file))
    if not m:
        return ""
    institution, volume_id = m.group(1), m.group(2)
    best_kw = max(kw_counts, key=lambda k: kw_counts[k])
    q1 = urllib.parse.quote(best_kw)
    return f"https://babel.hathitrust.org/cgi/pt/search?id={institution}.{volume_id}&q1={q1}&sz=25&start=1&sort=seq&hl=true"

# --- Paths ---
input_dir = Path("input_data")
output_dir = Path("output_data")
output_dir.mkdir(exist_ok=True)

csv_files = sorted(input_dir.glob("*.csv"))
print(f"Found {len(csv_files)} input files: {[f.name for f in csv_files]}")

# --- Filter, annotate, and save ---
for csv_path in csv_files:
    df = pd.read_csv(csv_path)
    text_col = df["text"].astype(str)

    # Keep only rows that contain at least one keyword
    mask = text_col.str.contains(combined_pattern)
    filtered = df[mask].reset_index(drop=True)
    text_filtered = filtered["text"].astype(str)

    # One column per keyword: count of occurrences in the text (0 if absent)
    kw_col_names = []
    for kw, pat in kw_patterns.items():
        col = "kw_" + kw.replace(" ", "_")
        filtered[col] = text_filtered.apply(lambda t, p=pat: len(p.findall(t)))
        kw_col_names.append(col)

    # Summary: number of distinct keywords matched and a readable list
    filtered["keyword_count"] = filtered[kw_col_names].gt(0).sum(axis=1)
    filtered["keywords_found"] = filtered[kw_col_names].apply(
        lambda row: ", ".join(kw for kw, col in zip(WATER_KEYWORDS, kw_col_names) if row[col] > 0),
        axis=1
    )

    # HathiTrust search link: q1 = most present keyword, static search params
    filtered["document_link"] = filtered.apply(
        lambda row: make_hathitrust_link(
            row["source_file"],
            {kw: row[col] for kw, col in zip(WATER_KEYWORDS, kw_col_names)}
        ),
        axis=1
    )

    out_path = output_dir / f"{csv_path.stem}_water_filtered.csv"
    filtered.to_csv(out_path, index=False)
    print(f"{csv_path.name}: {len(df)} rows -> {len(filtered)} kept ({len(filtered)/len(df):.1%}) -> {out_path.name}")

Found 4 input files: ['blackwood_sci_topics_lime.csv', 'cook_sci_topics_lime.csv', 'isis_sci_topics_lime.csv', 'pmg_sci_topics_lime.csv']
blackwood_sci_topics_lime.csv: 322 rows -> 67 kept (20.8%) -> blackwood_sci_topics_lime_water_filtered.csv
cook_sci_topics_lime.csv: 423 rows -> 91 kept (21.5%) -> cook_sci_topics_lime_water_filtered.csv
isis_sci_topics_lime.csv: 402 rows -> 29 kept (7.2%) -> isis_sci_topics_lime_water_filtered.csv
pmg_sci_topics_lime.csv: 303 rows -> 27 kept (8.9%) -> pmg_sci_topics_lime_water_filtered.csv


In [27]:
blackwood_df = pd.read_csv('output_data/blackwood_sci_topics_lime_water_filtered.csv')
cook_df = pd.read_csv('output_data/cook_sci_topics_lime_water_filtered.csv')
isis_df = pd.read_csv('output_data/isis_sci_topics_lime_water_filtered.csv')
pmg_df = pd.read_csv('output_data/pmg_sci_topics_lime_water_filtered.csv')

print(f"Blackwood articles: {len(blackwood_df)}")
print(f"Cook articles: {len(cook_df)}")
print(f"Isis articles: {len(isis_df)}")
print(f"PMG articles: {len(pmg_df)}")


Blackwood articles: 67
Cook articles: 91
Isis articles: 29
PMG articles: 27


In [28]:
# Article count per keyword per collection (articles can count for multiple keywords)
collections = {
    "Blackwood": blackwood_df,
    "Cook":      cook_df,
    "Isis":      isis_df,
    "PMG":       pmg_df,
}

rows = []
for kw in WATER_KEYWORDS:
    col = "kw_" + kw.replace(" ", "_")
    row = {"keyword": kw}
    for name, df in collections.items():
        row[name] = int((df[col] > 0).sum()) if col in df.columns else 0
    rows.append(row)

counts_df = pd.DataFrame(rows).set_index("keyword")
counts_df

,Blackwood,Cook,Isis,PMG
keyword,,,,
shore,6,0,0,2
ocean,15,26,17,2
sea,38,40,9,11
river,21,10,2,4
lake,3,6,0,5
pond,2,3,1,4
stream,11,20,3,3
creek,0,0,0,0
wetland,0,0,0,0


In [29]:
def top_by_keyword(df, n=20):
    result = {}
    for kw in WATER_KEYWORDS:
        col = "kw_" + kw.replace(" ", "_")
        if col not in df.columns:
            continue
        kw_df = (
            df[df[col] > 0]
            .sort_values([col, "keyword_count"], ascending=[False, False])
            .head(n)
            .reset_index(drop=True)
        )
        kw_df.index += 1  # 1-based rank
        if not kw_df.empty:
            result[kw] = kw_df
    return result


blackwood_top20 = top_by_keyword(blackwood_df)
cook_top20      = top_by_keyword(cook_df)
isis_top20      = top_by_keyword(isis_df)
pmg_top20       = top_by_keyword(pmg_df)

# Summary: rows per keyword per dataset
for label, top20 in [("BLACKWOOD", blackwood_top20), ("COOK", cook_top20),
                     ("ISIS", isis_top20), ("PMG", pmg_top20)]:
    print(f"\n{label}:")
    for kw, kw_df in top20.items():
        print(f"  {kw:20s}: {len(kw_df)} rows")


BLACKWOOD:
  shore               : 6 rows
  ocean               : 15 rows
  sea                 : 20 rows
  river               : 20 rows
  lake                : 3 rows
  pond                : 2 rows
  stream              : 11 rows
  bay                 : 3 rows

COOK:
  ocean               : 20 rows
  sea                 : 20 rows
  river               : 10 rows
  lake                : 6 rows
  pond                : 3 rows
  stream              : 20 rows
  reservoir           : 2 rows
  gulf                : 1 rows
  bay                 : 4 rows

ISIS:
  ocean               : 17 rows
  sea                 : 9 rows
  river               : 2 rows
  pond                : 1 rows
  stream              : 3 rows

PMG:
  shore               : 2 rows
  ocean               : 2 rows
  sea                 : 11 rows
  river               : 4 rows
  lake                : 5 rows
  pond                : 4 rows
  stream              : 3 rows
  bay                 : 3 rows
  body of water       : 2 ro

In [30]:
blackwood_top20

{'shore':    half_page_number                                               text  \
 1               310  ious countenance,\nThe shadows of the clouds t...   
 2               801  w hundred yards' start of\nDr Warburton, is se...   
 3                49  what depth, what\nexpanse, lay before me! How ...   
 4               298  truded\nhimself on my hook, which I contend he...   
 5              1953  his objection might be removed. It is on all h...   
 6              1252  curling column; then my eye would\ncatch the g...   
 
    predicted_label  science_prob  \
 1                1      0.992632   
 2                1      0.990180   
 3                1      0.993685   
 4                1      0.992309   
 5                1      0.991767   
 6                1      0.992902   
 
                                          source_file  \
 1  blackwood_v24_1828_inu-30000080770807-17467396...   
 2  blackwood_v24_1828_inu-30000080770807-17467396...   
 3  blackwood_v54_1843_inu-30000

In [31]:
def top20_to_df(top20_dict):
    frames = []
    for kw, kw_df in top20_dict.items():
        chunk = kw_df.reset_index().rename(columns={"index": "rank"})
        chunk.insert(0, "keyword", kw)
        frames.append(chunk)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


blackwood_top20_df = top20_to_df(blackwood_top20)
cook_top20_df      = top20_to_df(cook_top20)
isis_top20_df      = top20_to_df(isis_top20)
pmg_top20_df       = top20_to_df(pmg_top20)

In [32]:
blackwood_top20_df

,keyword,rank,half_page_number,text,predicted_label,science_prob,source_file,clean_text,llm_topic,kw_shore,...,kw_wetland,kw_reservoir,kw_estuary,kw_gulf,kw_bay,kw_body_of_water,kw_bodies_of_water,keyword_count,keywords_found,document_link
0,shore,1,310,"ious countenance,\nThe shadows of the clouds t...",1,0.992632,blackwood_v24_1828_inu-30000080770807-17467396...,ious countenance the shadows of the clouds tha...,natural history,1,...,0,0,0,0,0,0,0,3,"shore, ocean, sea",https://babel.hathitrust.org/cgi/pt/search?id=...
1,shore,2,801,"w hundred yards' start of\nDr Warburton, is se...",1,0.990180,blackwood_v24_1828_inu-30000080770807-17467396...,w hundred yards start of dr warburton is seen ...,fishing,1,...,0,0,0,0,0,0,0,2,"shore, sea",https://babel.hathitrust.org/cgi/pt/search?id=...
2,shore,3,49,"what depth, what\nexpanse, lay before me! How ...",1,0.993685,blackwood_v54_1843_inu-30000080771417-17467397...,what depth what expanse lay before me how sing...,oceanography,1,...,0,0,0,0,0,0,0,2,"shore, ocean",https://babel.hathitrust.org/cgi/pt/search?id=...
3,shore,4,298,"truded\nhimself on my hook, which I contend he...",1,0.992309,blackwood_v54_1843_inu-30000080771417-17467397...,truded himself on my hook which i contend he h...,fishery,1,...,0,0,0,0,0,0,0,2,"shore, river",https://babel.hathitrust.org/cgi/pt/search?id=...
4,shore,5,1953,his objection might be removed. It is on all h...,1,0.991767,blackwood_v54_1843_inu-30000080771417-17467397...,his objection might be removed it is on all ha...,commerce,1,...,0,0,0,0,0,0,0,2,"shore, sea",https://babel.hathitrust.org/cgi/pt/search?id=...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,stream,10,1840,", and wad\nscarcely do that for man, which nat...",1,0.993972,blackwood_v24_1828_inu-30000080770807-17467396...,and wad scarcely do that for man which natur a...,music,0,...,0,0,0,0,0,0,0,1,stream,https://babel.hathitrust.org/cgi/pt/search?id=...
76,stream,11,1263,hat lies beneath. Wild flowers grow\nall aroun...,1,0.992567,blackwood_v61_1847_inu-30000080764040-17467399...,hat lies beneath wild flowers grow all around ...,wildlife,0,...,0,0,0,0,0,0,0,1,stream,https://babel.hathitrust.org/cgi/pt/search?id=...
77,bay,1,2334,of Ameri-\nca is bestowed by the canvass-back\...,1,0.994884,blackwood_v31_1832_inu-30000080770757-17467396...,of ameri ca is bestowed by the canvassback duc...,natural history,0,...,0,0,0,0,1,0,0,1,bay,https://babel.hathitrust.org/cgi/pt/search?id=...
78,bay,2,617,"than he has hitherto seen, and\nthat now it i...",1,0.992593,blackwood_v61_1847_inu-30000080764040-17467399...,than he has hitherto seen and that now it is a...,curiosity,0,...,0,0,0,0,1,0,0,1,bay,https://babel.hathitrust.org/cgi/pt/search?id=...


In [33]:
# Save each to CSV
for name, df in [("blackwood", blackwood_top20_df), ("cook", cook_top20_df),
                 ("isis", isis_top20_df), ("pmg", pmg_top20_df)]:
    df.to_csv(output_dir / f"{name}_top20_by_keyword.csv", index=False)
    print(f"{name}: {len(df)} total rows across {df['keyword'].nunique()} keywords")

blackwood: 80 total rows across 8 keywords
cook: 86 total rows across 9 keywords
isis: 32 total rows across 5 keywords
pmg: 36 total rows across 9 keywords
